# Step1 Template: QC + Preprocessing with scLucid

This notebook is a cross-project Step1 template for single-cell RNA-seq preprocessing.

Design goals:
- Use `scLucid` high-level workflows by default.
- Keep `sample_key` as the minimal metadata contract.
- Enable sample-aware QC, custom HVG, and integration only when the data support them.
- Preserve reviewer-facing traceability in `adata.uns['sclucid']`.
- Avoid project-specific hard coding.


## Notes

- Do not pre-filter genes before cell QC unless you have a specific reason.
- `sample_key` should be retained even for single-sample projects. If missing, this template creates a synthetic sample label.
- Batch correction is disabled automatically when the batch key is obviously confounded with provided biology columns.
- Protein-coding filtering is optional and is not applied until after QC.


In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import scanpy as sc
import scLucid as scl

warnings.filterwarnings('ignore')

scl.setup_logging('INFO')
scl.set_figure_params(
    dpi=150,
    dpi_save=300,
    figsize=(6, 5),
    style='seaborn-v0_8',
    style_dict={'axes.grid': True},
    color_theme='default',
)


In [ ]:
# -----------------------------
# User parameters
# -----------------------------
PROJECT_DIR = Path('..').resolve()
RAW_H5AD = PROJECT_DIR / 'data' / 'raw_counts.h5ad'
METADATA_CSV = None
METADATA_INDEX_COL = None

OUTPUT_DIR = PROJECT_DIR / 'results' / 'step1_template'
QC_DIR = OUTPUT_DIR / 'qc'
PP_DIR = OUTPUT_DIR / 'preprocess'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPECIES = 'human'
TISSUE = None
TISSUE_TYPE = 'unknown'

SAMPLE_KEY = 'sampleID'
BIOLOGY_COLUMNS = []

# Optional metadata merge behavior.
# If METADATA_CSV is provided, it should contain row-level metadata keyed by cell barcode.
MERGE_METADATA = True

# Optional gene biotype handling after QC.
RUN_BIOTYPE_FILTER = False
BIOTYPE_METHOD = 'reference'  # 'reference' or 'custom'; legacy 'ensembl' is still accepted
BIOTYPE_ALLOW_DOWNLOAD = True
BIOTYPE_CACHE_DIR = None
BIOTYPE_PREFER_BUNDLED = True
CUSTOM_BIOTYPE_TABLE = None
KEEP_BIOTYPES = ['protein_coding']

# Preprocess behavior.
PREFER_CUSTOM_HVG_FOR_MULTI_SAMPLE = True
RUN_INTEGRATION = 'auto'  # 'auto', True, False
INTEGRATION_METHOD = 'harmony'
N_TOP_GENES = 2000

# Optional post-workflow embedding optimization.
RUN_EMBEDDING_OPTIMIZATION = True
OPTIMIZATION_N_NEIGHBORS = [15, 20, 30]
OPTIMIZATION_N_PCS = [20, 30, 40, 50]


In [ ]:
def ensure_counts_layer(adata):
    if 'counts' not in adata.layers:
        adata.layers['counts'] = adata.X.copy()
    return adata


def load_optional_metadata():
    if METADATA_CSV is None:
        return None
    metadata = pd.read_csv(METADATA_CSV, index_col=METADATA_INDEX_COL)
    metadata.index = metadata.index.astype(str)
    return metadata


def merge_cell_metadata(adata, metadata):
    if metadata is None or not MERGE_METADATA:
        return adata
    shared = adata.obs_names.intersection(metadata.index)
    if len(shared) == 0:
        raise ValueError('No overlapping cell barcodes between adata.obs_names and metadata index.')
    for col in metadata.columns:
        adata.obs[col] = metadata.reindex(adata.obs_names)[col].values
    return adata


def ensure_sample_key(adata, sample_key):
    if sample_key in adata.obs.columns:
        adata.obs[sample_key] = adata.obs[sample_key].astype(str)
        return adata

    candidate_keys = [
        'sampleID', 'sample', 'Sample', 'orig.ident', 'orig_ident',
        'patient', 'patient_id', 'donor', 'donor_id', 'batch', 'Batch'
    ]
    for candidate in candidate_keys:
        if candidate in adata.obs.columns:
            adata.obs[sample_key] = adata.obs[candidate].astype(str)
            print(f"Using existing obs column '{candidate}' as '{sample_key}'.")
            return adata

    adata.obs[sample_key] = 'sample_1'
    print(f"'{sample_key}' not found. Created synthetic single-sample labels.")
    return adata


def summarize_metadata(adata, sample_key, biology_columns):
    print(f'Cells: {adata.n_obs:,}')
    print(f'Genes: {adata.n_vars:,}')
    print(f"Samples ({sample_key}): {adata.obs[sample_key].nunique()}")
    display(adata.obs[[c for c in [sample_key] + biology_columns if c in adata.obs.columns]].head())


def should_use_custom_hvg(adata, sample_key):
    if not PREFER_CUSTOM_HVG_FOR_MULTI_SAMPLE:
        return False
    if sample_key not in adata.obs.columns:
        return False
    n_samples = adata.obs[sample_key].nunique()
    min_cells = adata.obs[sample_key].value_counts().min()
    return n_samples >= 2 and min_cells >= 50


def detect_integration_confounding(adata, batch_key, biology_columns):
    if batch_key not in adata.obs.columns:
        return ['missing_batch_key']

    problems = []
    for col in biology_columns:
        if col not in adata.obs.columns:
            continue
        mapping = adata.obs[[batch_key, col]].dropna().drop_duplicates()
        per_batch = mapping.groupby(batch_key)[col].nunique()
        if len(per_batch) > 0 and (per_batch <= 1).all():
            n_batch = adata.obs[batch_key].nunique()
            n_col = adata.obs[col].nunique()
            if n_batch == n_col:
                problems.append(f'{batch_key} is one-to-one with {col}')
    return problems


def resolve_integration_flag(adata, batch_key, biology_columns):
    if RUN_INTEGRATION is True:
        return True, []
    if RUN_INTEGRATION is False:
        return False, []

    if batch_key not in adata.obs.columns or adata.obs[batch_key].nunique() < 2:
        return False, ['not multi-sample']

    problems = detect_integration_confounding(adata, batch_key, biology_columns)
    return len(problems) == 0, problems


## Load Data

In [ ]:
adata = sc.read_h5ad(RAW_H5AD)
adata = ensure_counts_layer(adata)

metadata = load_optional_metadata()
adata = merge_cell_metadata(adata, metadata)
adata = ensure_sample_key(adata, SAMPLE_KEY)

for col in [SAMPLE_KEY] + BIOLOGY_COLUMNS:
    if col in adata.obs.columns:
        adata.obs[col] = adata.obs[col].astype(str)

summarize_metadata(adata, SAMPLE_KEY, BIOLOGY_COLUMNS)


## Run QC

This template uses `run_standard_qc()` so that sample-aware thresholding, reviewer-facing summaries, and package traceability are preserved.

In [ ]:
qc_config = scl.qc.QCWorkflowConfig(
    sample_key=SAMPLE_KEY,
    species=SPECIES,
    tissue=TISSUE,
    tissue_type=TISSUE_TYPE,
    threshold_mode='hierarchical',
    use_recommendations=True,
    save_dir=str(QC_DIR),
    metrics_reporting_config=scl.qc.MetricsReportingConfig(
        show_plots=True,
        plot_violin=True,
        plot_scatter=True,
        export_stats=True,
    ),
    doublet_config=scl.qc.DoubletConfig(
        method='scrublet',
        marker_species=SPECIES,
        marker_tissue=TISSUE,
        use_heuristics=True,
        plot_summary=True,
        plot_bar=True,
        plot_scatter=True,
        show_plots=True,
    ),
)

adata_qc = scl.run_standard_qc(
    adata,
    config=qc_config,
    tissue_type=TISSUE_TYPE,
    show_progress=True,
)

print(f'Cells after QC: {adata_qc.n_obs:,}')
print(f'Genes after QC: {adata_qc.n_vars:,}')
adata_qc.write_h5ad(OUTPUT_DIR / 'step1_qc_filtered.h5ad', compression='gzip')


## Optional Gene Biotype Filtering

Default recommendation:
- Keep this step off unless the project explicitly requires protein-coding-only analysis.
- Prefer package-managed annotation (`method='reference'`) over a project-specific local file.
- For `method='reference'`, scLucid now resolves biotype references in this order:
  package resource -> local cache -> one-time download.
- If a bundled or cached table is available, no network access is needed.


In [ ]:
adata_pp_input = adata_qc.copy()

if RUN_BIOTYPE_FILTER:
    print('Gene biotype resource discovery:')
    print(scl.pp.list_gene_biotype_resources(species=SPECIES, cache_dir=BIOTYPE_CACHE_DIR))

    adata_pp_input = scl.pp.apply_gene_biotype_strategy(
        adata_pp_input,
        species=SPECIES,
        method=BIOTYPE_METHOD,
        custom_biotype_path=CUSTOM_BIOTYPE_TABLE,
        keep_biotypes=KEEP_BIOTYPES,
        do_filter=True,
        fuzzy_match=True,
        overwrite=True,
        allow_download=BIOTYPE_ALLOW_DOWNLOAD,
        cache_dir=BIOTYPE_CACHE_DIR,
        prefer_bundled=BIOTYPE_PREFER_BUNDLED,
        copy=True,
    )

    print('Biotype filtering applied.')
    print(adata_pp_input.var['biotype_category'].value_counts().head())
    print(adata_pp_input.uns['sclucid']['preprocess']['gene_biotypes'])
else:
    print('Skipping gene biotype filtering.')

adata_pp_input = ensure_counts_layer(adata_pp_input)


## Configure Preprocessing

The template prefers:
- `custom` HVG for multi-sample data with enough cells per sample.
- `scanpy` HVG for single-sample or small multi-sample data.
- Integration only when multi-sample and not obviously confounded with supplied biology columns.


In [ ]:
use_custom_hvg = should_use_custom_hvg(adata_pp_input, SAMPLE_KEY)
run_integration, integration_warnings = resolve_integration_flag(adata_pp_input, SAMPLE_KEY, BIOLOGY_COLUMNS)

hvg_config = scl.pp.HVGConfig(
    method='custom' if use_custom_hvg else 'scanpy',
    n_top_genes=N_TOP_GENES,
    flavor='seurat',
    sample_key=SAMPLE_KEY,
    min_n_samples=2,
    n_highly_expressed_genes=50,
    n_specific_genes=10,
)

pp_config = scl.pp.WorkflowConfig(
    save_dir=str(PP_DIR),
    hvg=hvg_config,
    scaling=scl.pp.ScalingConfig(
        vars_to_regress=['total_counts', 'pct_counts_mt', 'S_score', 'G2M_score'],
        regress_in_scale=False,
        max_value=10,
    ),
    integration=scl.pp.IntegrationConfig(
        method=INTEGRATION_METHOD if run_integration else None,
        batch_key=SAMPLE_KEY if run_integration else None,
        use_rep='X_pca',
    ),
    graph=scl.pp.GraphConfig(
        n_pcs=50,
        n_neighbors=15,
    ),
    run_regression=True,
    run_scaling=True,
    run_pca=True,
    run_neighbors=True,
    run_integration=run_integration,
)

print(f'Custom HVG enabled: {use_custom_hvg}')
print(f'Integration enabled: {run_integration}')
if integration_warnings:
    print('Integration warnings:')
    for warning in integration_warnings:
        print(f'  - {warning}')


In [ ]:
adata_pp = scl.run_preprocessing(
    adata_pp_input,
    config=pp_config,
    tissue_type=TISSUE_TYPE,
    show_progress=True,
    keep_intermediate_layers=True,
)

print(f'Cells after preprocessing: {adata_pp.n_obs:,}')
print(f'Genes after preprocessing: {adata_pp.n_vars:,}')
print(f"Available embeddings: {[k for k in adata_pp.obsm.keys() if k.startswith('X_')]}")

adata_pp.write_h5ad(OUTPUT_DIR / 'step1_preprocessed.h5ad', compression='gzip')


## Optional Embedding Optimization

This cell is optional. It uses `scLucid.pp.optimize_neighbors_pcs()` on the final representation and then rebuilds neighbors/UMAP with the best-scoring setting.

In [ ]:
if run_integration and 'X_harmony' in adata_pp.obsm:
    final_rep = 'X_harmony'
else:
    final_rep = 'X_pca'

if RUN_EMBEDDING_OPTIMIZATION:
    optimization_results = scl.pp.optimize_neighbors_pcs(
        adata_pp,
        config=scl.pp.NeighborsConfig(
            use_rep=final_rep,
            n_neighbors_list=OPTIMIZATION_N_NEIGHBORS,
            n_pcs_list=OPTIMIZATION_N_PCS,
            n_jobs=-1,
        ),
    )

    if len(optimization_results) > 0:
        best_row = optimization_results.loc[optimization_results['silhouette_score'].idxmax()]
        best_n_neighbors = int(best_row['n_neighbors'])
        best_n_pcs = int(best_row['n_pcs'])
        print(f'Using optimized parameters: n_neighbors={best_n_neighbors}, n_pcs={best_n_pcs}')

        sc.pp.neighbors(adata_pp, use_rep=final_rep, n_neighbors=best_n_neighbors, n_pcs=best_n_pcs)
        sc.tl.umap(adata_pp)
        adata_pp.uns.setdefault('sclucid', {}).setdefault('preprocess', {})['final_graph'] = {
            'use_rep': final_rep,
            'n_neighbors': best_n_neighbors,
            'n_pcs': best_n_pcs,
        }
    else:
        print('No valid optimization result returned; keeping workflow default graph.')
else:
    print('Skipping embedding optimization.')

adata_pp.write_h5ad(OUTPUT_DIR / 'step1_preprocessed_final.h5ad', compression='gzip')


## Quick Checks

In [ ]:
plot_cols = [c for c in [SAMPLE_KEY] + BIOLOGY_COLUMNS if c in adata_pp.obs.columns]
if len(plot_cols) == 0:
    plot_cols = None

if plot_cols is not None:
    sc.pl.umap(adata_pp, color=plot_cols, wspace=0.4)

adata_pp


## Outputs

- `step1_qc_filtered.h5ad`: QC-filtered AnnData.
- `step1_preprocessed.h5ad`: workflow output after standard preprocessing.
- `step1_preprocessed_final.h5ad`: final object after optional embedding optimization.
- `results/step1_template/qc`: reviewer-facing QC artifacts.
- `results/step1_template/preprocess`: preprocessing artifacts.
